# Experiment 4 analysis

Reads the consolidated CSVs under `data/outputs/Experiment 4/_analysis/`. Run-level tables come from `runs_cache.csv`; score plots come from `evaluation_scores.csv`.

The notebook does not read raw experiment output folders.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks and aux scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hardware_equivalence import normalize_latency

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams["figure.dpi"] = 110

ALL_MODELS = [
    "patchcore", "padim", "subspacead", "stfpm",
    "csflow", "draem", "rd4ad",
]

POLYMER_SHEET_CATEGORIES = ["Polymer_sheet"]

EXP_ROOT = PROJECT_ROOT / "data" / "outputs" / "Experiment 4"
ANALYSIS_DIR = EXP_ROOT / "_analysis"
RUNS_CSV = ANALYSIS_DIR / "runs_cache.csv"
SCORES_CSV = ANALYSIS_DIR / "evaluation_scores.csv"

print(f"Runs CSV: {RUNS_CSV}")
print(f"Scores CSV: {SCORES_CSV}")


In [ ]:
df = pd.read_csv(RUNS_CSV)
score_df = pd.read_csv(SCORES_CSV, low_memory=False)
normalize_latency(df)
MODELS_PRESENT = sorted(df["model"].dropna().unique()) if not df.empty else []
print(f"Loaded {len(df)} consolidated run rows across models: {MODELS_PRESENT}")
print(f"Loaded {len(score_df)} consolidated evaluation score rows.")


## 1. Runs per category × model

Count of output folders found in `Experiment 4/` for each (category, model) pair.

In [ ]:
runs_per_cat = (
    df.groupby(["category", "model"]).size().unstack(fill_value=0)
    .reindex(index=POLYMER_SHEET_CATEGORIES, columns=ALL_MODELS, fill_value=0)
)
runs_per_cat.index.name = "category"
runs_per_cat.loc["TOTAL"] = runs_per_cat.sum(axis=0)
runs_per_cat

## 2. Per-model averaged metrics

For each model, mean across the categories actually run. Columns include every model supported by the pipeline; models with no Experiment 4 outputs show NaN.

In [ ]:
METRICS_AVG = ["auroc", "aupr", "precision", "recall", "mean_latency_ms", "f1"]

model_avg = df.groupby("model")[METRICS_AVG].mean(numeric_only=True).T
model_avg = model_avg.reindex(columns=ALL_MODELS)
model_avg.index.name = "metric"
model_avg

## 3. Category × model — mini-matrix per cell

Each cell is a 2×2 block:

| AUROC | F1 |
| --- | --- |
| **AUPR** | **latency (ms)** |

In [ ]:
def _fmt(v: float, is_latency: bool = False) -> str:
    if pd.isna(v):
        return "\u2014"
    return f"{v:.1f}" if is_latency else f"{v:.3f}"


def render_minimatrix(df: pd.DataFrame, models: list[str], categories: list[str]) -> str:
    aur = df.pivot_table(index="category", columns="model", values="auroc")
    f1m = df.pivot_table(index="category", columns="model", values="f1")
    aup = df.pivot_table(index="category", columns="model", values="aupr")
    lat = df.pivot_table(index="category", columns="model", values="mean_latency_ms")

    def get(pivot, c, m):
        if c in pivot.index and m in pivot.columns:
            return pivot.loc[c, m]
        return float("nan")

    def cell_html(c: str, m: str) -> str:
        cell = (
            "<table style='border-collapse:collapse;font-size:10px;width:100%'>"
            "<tr>"
            f"<td style='padding:1px 4px;border-right:1px solid #ddd;border-bottom:1px solid #ddd'>"
            f"<span style='color:#888'>AUROC</span><br><b>{_fmt(get(aur, c, m))}</b></td>"
            f"<td style='padding:1px 4px;border-bottom:1px solid #ddd'>"
            f"<span style='color:#888'>F1</span><br><b>{_fmt(get(f1m, c, m))}</b></td>"
            "</tr><tr>"
            f"<td style='padding:1px 4px;border-right:1px solid #ddd'>"
            f"<span style='color:#888'>AUPR</span><br><b>{_fmt(get(aup, c, m))}</b></td>"
            f"<td style='padding:1px 4px'>"
            f"<span style='color:#888'>LAT ms</span><br><b>{_fmt(get(lat, c, m), is_latency=True)}</b></td>"
            "</tr></table>"
        )
        return cell

    header = "".join(
        f"<th style='border:1px solid #999;padding:4px;background:#f3f3f3'>{m}</th>"
        for m in models
    )
    body = []
    for c in categories:
        cells_html = "".join(
            f"<td style='border:1px solid #999;padding:2px;vertical-align:top'>{cell_html(c, m)}</td>"
            for m in models
        )
        body.append(
            "<tr>"
            f"<th style='border:1px solid #999;padding:4px;text-align:left;background:#f3f3f3'>{c}</th>"
            f"{cells_html}</tr>"
        )
    return (
        "<table style='border-collapse:collapse'>"
        f"<tr><th style='border:1px solid #999;padding:4px;background:#f3f3f3'>category</th>{header}</tr>"
        + "".join(body)
        + "</table>"
    )


display(HTML(render_minimatrix(df, ALL_MODELS, POLYMER_SHEET_CATEGORIES)))

## 4. Per-model detailed table

One table per model. Rows are categories that were run for that model; columns are the headline run-level metrics.

In [ ]:
PER_MODEL_COLS = [
    "threshold_value",
    "warmup_frames", "calibration_frames", "streaming_frames",
    "global_auroc", "precision", "recall", "f1", "accuracy",
    "mean_score_ok", "mean_score_ng",
    "mean_latency_ms", "throughput_fps",
]

for model in MODELS_PRESENT:
    sub = (
        df[df["model"] == model]
        .set_index("category")[PER_MODEL_COLS]
        .sort_index()
    )
    print(f"\n=== {model} ({len(sub)} runs) ===")
    display(sub)

## 5. Heatmap — AUROC (category × model)

In [ ]:
def metric_heatmap(metric: str, title: str, *, vmin: float = 0.0, vmax: float = 1.0) -> None:
    mat = (
        df.pivot_table(index="category", columns="model", values=metric)
        .reindex(index=POLYMER_SHEET_CATEGORIES, columns=ALL_MODELS)
    )
    fig, ax = plt.subplots(
        figsize=(1.1 * len(ALL_MODELS) + 2, 0.32 * len(POLYMER_SHEET_CATEGORIES) + 1.5)
    )
    masked = np.ma.masked_invalid(mat.values)
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="#e5e5e5")
    im = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(ALL_MODELS)))
    ax.set_xticklabels(ALL_MODELS, rotation=45, ha="right")
    ax.set_yticks(range(len(POLYMER_SHEET_CATEGORIES)))
    ax.set_yticklabels(POLYMER_SHEET_CATEGORIES)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if pd.notna(v):
                ax.text(
                    j, i, f"{v:.2f}",
                    ha="center", va="center", fontsize=7,
                    color="white" if v < (vmin + vmax) / 2 else "black",
                )
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    plt.show()


metric_heatmap("auroc", "AUROC \u2014 category \u00d7 model")

## 6. Heatmap — F1 (category × model)

In [ ]:
metric_heatmap("f1", "F1 \u2014 category \u00d7 model")

## 7. Streaming scores - OK vs NG per (model, category)

Scatter of the per-frame anomaly score over the *evaluation* phase only (warm-up and threshold-calibration frames are excluded).

- **Y axis** is the score **min-max normalized to [0, 1] per plot** (the dashed threshold line is normalized with the same min/max so its position is meaningful within the plot).
- **X axis** keeps the original online visualization: frame order within the evaluation phase.

Green = OK ground truth, red = NG.


In [ ]:
SCATTER_COLS = 3

for model in MODELS_PRESENT:
    sub = df[df["model"] == model].sort_values("category").reset_index(drop=True)
    if sub.empty:
        continue
    n = len(sub)
    nrows = math.ceil(n / SCATTER_COLS)
    fig, axes = plt.subplots(
        nrows, SCATTER_COLS,
        figsize=(SCATTER_COLS * 4.2, nrows * 2.8),
        squeeze=False,
    )
    fig.suptitle(f"{model} - streaming scores (OK vs NG) per category", fontsize=12)
    for i, row in sub.iterrows():
        ax = axes[i // SCATTER_COLS][i % SCATTER_COLS]
        recs = score_df[
            (score_df["experiment"] == row["experiment"])
            & (score_df["model"] == row["model"])
        ].sort_values("idx")
        if recs.empty:
            ax.set_visible(False)
            continue
        idx = recs["idx"].to_numpy(dtype=int)
        scores = recs["score"].to_numpy(dtype=float)
        labels = recs["true_label"].to_numpy()

        finite = np.isfinite(scores)
        if not finite.any():
            ax.set_visible(False)
            continue
        s_min = float(scores[finite].min())
        s_max = float(scores[finite].max())
        denom = (s_max - s_min) if s_max > s_min else 1.0
        scores_norm = (scores - s_min) / denom

        ok = labels == 0
        ng = labels == 1
        ax.scatter(idx[ok], scores_norm[ok], s=8, c="tab:green", alpha=0.55, label="OK")
        ax.scatter(idx[ng], scores_norm[ng], s=8, c="tab:red", alpha=0.55, label="NG")
        thr = row["threshold_value"]
        if pd.notna(thr):
            thr_norm = (float(thr) - s_min) / denom
            ax.axhline(thr_norm, color="black", lw=0.7, ls="--", label=f"thr={thr_norm:.2f}")
        ax.set_title(row["category"], fontsize=9)
        ax.set_xlabel("frame order (evaluation idx)", fontsize=8)
        ax.set_ylabel("score (norm 0..1)", fontsize=8)
        ax.set_ylim(-0.05, 1.05)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, loc="best")
        ax.grid(alpha=0.25)
    for j in range(n, nrows * SCATTER_COLS):
        axes[j // SCATTER_COLS][j % SCATTER_COLS].set_visible(False)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


## 8. Online (Exp 4) vs Offline (Exp 2) — variación de métricas

Comparación del approach **online streaming** (Exp 4) contra el approach **offline** (Exp 2) para los mismos modelos sobre el dataset Polymer_sheet (una sola categoría). Para cada métrica se muestran:

- `offline` — valor de baseline (Exp 2);
- `online` — valor en streaming (Exp 4);
- `Δ = online − offline`.

Para AUROC, AUPR y F1 mayor es mejor: **verde = mejora**, **rojo = degradación**. Para `latency_ms` menor es mejor, por lo que el coloreo se invierte: **rojo = más lento (peor)**, **verde = más rápido**.


In [ ]:
OFFLINE_BASELINE_CSV = PROJECT_ROOT / "data" / "outputs" / "Experiment 2" / "_analysis" / "runs_cache.csv"
POLYMER_CATEGORY = "Polymer_sheet"

offline_df = pd.read_csv(OFFLINE_BASELINE_CSV)
normalize_latency(offline_df)
offline_df["category"] = POLYMER_CATEGORY
offline_df = offline_df.groupby(["model", "category"], as_index=False)[
    ["auroc", "aupr", "f1", "mean_latency_ms"]
].mean(numeric_only=True)
print(f"Offline baseline (Exp 2): {len(offline_df)} (model, category) rows across "
      f"models: {sorted(offline_df['model'].unique()) if not offline_df.empty else []}.")

offline_idx = offline_df.set_index(["model", "category"]) if not offline_df.empty else pd.DataFrame()


def offline_value(model: str, category: str, metric: str) -> float:
    if offline_idx.empty:
        return float("nan")
    try:
        v = offline_idx.loc[(model, category)][metric]
        return float(v) if pd.notna(v) else float("nan")
    except KeyError:
        return float("nan")


In [ ]:
COMPARE_METRICS = ["auroc", "aupr", "f1", "mean_latency_ms"]
COMPARE_METRIC_LABEL = {
    "auroc": "AUROC", "aupr": "AUPR", "f1": "F1", "mean_latency_ms": "Latency (ms)",
}
# Lower is better only for latency. Δ-color is inverted there.
LOWER_IS_BETTER = {"mean_latency_ms"}


def _delta_color(metric: str, v: float) -> str:
    if pd.isna(v) or v == 0:
        return ""
    improvement = (v < 0) if metric in LOWER_IS_BETTER else (v > 0)
    return "color:#0a7d2c;font-weight:bold" if improvement else "color:#b00020;font-weight:bold"


def _fmt_metric(metric: str, v: float, signed: bool = False) -> str:
    if pd.isna(v):
        return "\u2014"
    if metric == "mean_latency_ms":
        return f"{v:+.1f}" if signed else f"{v:.1f}"
    return f"{v:+.3f}" if signed else f"{v:.3f}"


In [ ]:
def render_summary_delta_table(online_df: pd.DataFrame, models: list[str], *, exp_label: str = "Exp 4 vs Exp 2") -> str:
    header_models = "".join(
        f"<th colspan='3' style='border:1px solid #999;padding:4px;background:#f3f3f3;text-align:center'>{m}</th>"
        for m in models
    )
    sub_header = "".join(
        "<th style='border:1px solid #999;padding:3px;background:#fafafa;font-size:11px'>offline</th>"
        "<th style='border:1px solid #999;padding:3px;background:#fafafa;font-size:11px'>online</th>"
        "<th style='border:1px solid #999;padding:3px;background:#fafafa;font-size:11px'>\u0394</th>"
        for _ in models
    )
    rows_html = []
    for metric in COMPARE_METRICS:
        cells = []
        for model in models:
            on_sub = online_df[online_df["model"] == model]
            cats = sorted(on_sub["category"].dropna().unique())
            online_vals = on_sub[metric].dropna()
            online_mean = float(online_vals.mean()) if not online_vals.empty else float("nan")
            off_vals = [offline_value(model, c, metric) for c in cats]
            off_vals = [v for v in off_vals if pd.notna(v)]
            offline_mean = float(np.mean(off_vals)) if off_vals else float("nan")
            delta = (online_mean - offline_mean) if (pd.notna(online_mean) and pd.notna(offline_mean)) else float("nan")
            cells.append(
                f"<td style='border:1px solid #999;padding:3px;text-align:right;color:#555;font-size:11px'>{_fmt_metric(metric, offline_mean)}</td>"
                f"<td style='border:1px solid #999;padding:3px;text-align:right;font-size:11px'>{_fmt_metric(metric, online_mean)}</td>"
                f"<td style='border:1px solid #999;padding:3px;text-align:right;font-size:11px;{_delta_color(metric, delta)}'>{_fmt_metric(metric, delta, signed=True)}</td>"
            )
        rows_html.append(
            f"<tr><th style='border:1px solid #999;padding:4px;background:#f3f3f3;text-align:left'>{COMPARE_METRIC_LABEL[metric]}</th>{''.join(cells)}</tr>"
        )
    return (
        f"<h4 style='margin:8px 0 4px'>Per-model averaged \u2014 online vs offline ({exp_label})</h4>"
        "<table style='border-collapse:collapse'>"
        f"<tr><th rowspan='2' style='border:1px solid #999;padding:4px;background:#f3f3f3'>metric</th>{header_models}</tr>"
        f"<tr>{sub_header}</tr>"
        + "".join(rows_html)
        + "</table>"
    )


display(HTML(render_summary_delta_table(df, MODELS_PRESENT, exp_label="Exp 4 vs Exp 2")))


In [ ]:
def render_per_category_delta_table(model: str, online_df: pd.DataFrame, *, exp_label: str = "Exp 4 vs Exp 2") -> str:
    sub = online_df[online_df["model"] == model]
    if sub.empty:
        return ""
    cats = sorted(sub["category"].dropna().unique())
    head_groups = "".join(
        f"<th colspan='3' style='border:1px solid #999;padding:4px;background:#f3f3f3;text-align:center'>{COMPARE_METRIC_LABEL[m]}</th>"
        for m in COMPARE_METRICS
    )
    sub_head = "".join(
        "<th style='border:1px solid #999;padding:3px;background:#fafafa;font-size:11px'>offline</th>"
        "<th style='border:1px solid #999;padding:3px;background:#fafafa;font-size:11px'>online</th>"
        "<th style='border:1px solid #999;padding:3px;background:#fafafa;font-size:11px'>\u0394</th>"
        for _ in COMPARE_METRICS
    )
    rows_html = []
    for cat in cats:
        cells = [f"<td style='border:1px solid #999;padding:3px;text-align:left'><b>{cat}</b></td>"]
        for metric in COMPARE_METRICS:
            base_v = offline_value(model, cat, metric)
            row = sub[sub["category"] == cat][metric]
            cur_v = float(row.iloc[0]) if not row.empty else float("nan")
            delta = (cur_v - base_v) if (pd.notna(cur_v) and pd.notna(base_v)) else float("nan")
            cells.append(
                f"<td style='border:1px solid #999;padding:3px;text-align:right;color:#555;font-size:11px'>{_fmt_metric(metric, base_v)}</td>"
                f"<td style='border:1px solid #999;padding:3px;text-align:right;font-size:11px'>{_fmt_metric(metric, cur_v)}</td>"
                f"<td style='border:1px solid #999;padding:3px;text-align:right;font-size:11px;{_delta_color(metric, delta)}'>{_fmt_metric(metric, delta, signed=True)}</td>"
            )
        rows_html.append(f"<tr>{''.join(cells)}</tr>")
    return (
        f"<h4 style='margin:8px 0 4px'>{model} \u2014 {exp_label} ({len(cats)} categories)</h4>"
        "<table style='border-collapse:collapse'>"
        "<tr><th rowspan='2' style='border:1px solid #999;padding:4px;background:#f3f3f3'>category</th>"
        f"{head_groups}</tr>"
        f"<tr>{sub_head}</tr>"
        + "".join(rows_html)
        + "</table>"
    )


for _model in MODELS_PRESENT:
    _html = render_per_category_delta_table(_model, df, exp_label="Exp 4 vs Exp 2")
    if _html:
        display(HTML(_html))
